In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

DATA = PROJECT_ROOT / "data" / "processed" / "preprocessed_data_2"

for f in sorted(DATA.glob('*.csv')):
    print(f"{f.name:55s} {f.stat().st_size/1024**2:8.1f} MB")

accounts_friendrequest_preprocessed_2.csv                  495.4 MB
accounts_paymenthistory_preprocessed_2.csv                   4.4 MB
accounts_pointhistory_preprocessed_2.csv                     4.5 MB
accounts_user_preprocessed_2.csv                            40.0 MB
accounts_userquestionrecord_preprocessed_2.csv              67.9 MB
polls_questionpiece_preprocessed_2.csv                      29.8 MB


In [2]:
user   = pd.read_csv(DATA / "accounts_user_preprocessed_2.csv")
pay    = pd.read_csv(DATA / "accounts_paymenthistory_preprocessed_2.csv")
ph     = pd.read_csv(DATA / "accounts_pointhistory_preprocessed_2.csv")
uqr    = pd.read_csv(DATA / "accounts_userquestionrecord_preprocessed_2.csv")
qpiece = pd.read_csv(DATA / "polls_questionpiece_preprocessed_2.csv")
fr     = pd.read_csv(DATA / "accounts_friendrequest_preprocessed_2.csv")

for name, df in [('user', user), ('payment', pay), ('pointhistory', ph),
                 ('uqr', uqr), ('questionpiece', qpiece), ('friendrequest', fr)]:
    print(f"\n{name}: {df.shape}")
    print(f"  {df.columns.tolist()}")


user: (677075, 11)
  ['id', 'gender', 'point', 'is_push_on', 'created_at', 'ban_status', 'report_count', 'alarm_count', 'pending_chat', 'pending_votes', 'group_id']

payment: (95137, 5)
  ['id', 'productId', 'phone_type', 'created_at', 'user_id']

pointhistory: (88762, 5)
  ['id', 'delta_point', 'created_at', 'user_id', 'user_question_record_id']

uqr: (800793, 12)
  ['id', 'status', 'created_at', 'chosen_user_id', 'question_id', 'user_id', 'question_piece_id', 'has_read', 'answer_status', 'answer_updated_at', 'report_count', 'opened_times']

questionpiece: (823739, 5)
  ['id', 'is_voted', 'created_at', 'question_id', 'is_skipped']

friendrequest: (7793140, 6)
  ['id', 'status', 'created_at', 'updated_at', 'receive_user_id', 'send_user_id']


In [3]:
for name, df, col in [('user', user, 'created_at'),
                      ('payment', pay, 'created_at'),
                      ('pointhistory', ph, 'created_at'),
                      ('uqr', uqr, 'created_at'),
                      ('questionpiece', qpiece, 'created_at'),
                      ('friendrequest', fr, 'created_at')]:
    if col in df.columns:
        s = pd.to_datetime(df[col])
        print(f"{name:15s} {s.min()} ~ {s.max()}")

user            2023-03-29 05:18:56.162368 ~ 2024-05-07 13:40:24.474102
payment         2023-05-13 21:28:34 ~ 2024-05-06 16:06:30
pointhistory    2023-05-13 00:00:44 ~ 2024-05-07 11:01:05
uqr             2023-05-13 00:00:02 ~ 2024-05-07 11:29:07
questionpiece   2023-05-13 00:00:04 ~ 2024-05-07 11:32:30
friendrequest   2023-05-13 00:00:00 ~ 2024-05-07 11:25:27


In [4]:
print(f"user: {len(user):,}")
print()
for name, df, col in [('friendrequest(send)', fr, 'send_user_id'),
                      ('friendrequest(recv)', fr, 'receive_user_id'),
                      ('payment', pay, 'user_id'),
                      ('pointhistory', ph, 'user_id'),
                      ('uqr(투표자)', uqr, 'user_id'),
                      ('uqr(수신자)', uqr, 'chosen_user_id')]:
    n = df[col].nunique()
    print(f"{name:22s} {n:>8,}명  {n/len(user)*100:5.2f}%")

user: 677,075

friendrequest(send)     475,793명  70.27%
friendrequest(recv)     508,356명  75.08%
payment                  59,191명   8.74%
pointhistory              4,021명   0.59%
uqr(투표자)                  4,294명   0.63%
uqr(수신자)                 13,811명   2.04%


In [8]:
step1 = set(user['id'])

fr_a = fr[fr['status'] == 'A']
friends = pd.concat([fr_a['send_user_id'], fr_a['receive_user_id']]).value_counts()
step2 = set(friends.index) & step1

step3 = step2 & set(uqr['chosen_user_id'])

step4 = step3 & set(pay['user_id'])

steps = [('가입', step1), ('친구 형성', step2),
         ('투표 수신', step3), ('첫 결제', step4)]

print(f"{'단계':18s} {'인원':>10s} {'전환율':>9s} {'도달률':>9s}")
print("-" * 52)

prev = None
for i, (name, s) in enumerate(steps):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    reach = f"{len(s)/len(step1)*100:7.2f}%"
    print(f"Step {i+1} {name:12s} {len(s):>10,} {conv:>9s} {reach:>9s}")
    prev = s

print()
for i in range(len(steps)-1):
    a, b = len(steps[i][1]), len(steps[i+1][1])
    print(f"{steps[i][0]:10s} → {steps[i+1][0]:10s} {a-b:>8,}명 ({(1-b/a)*100:5.1f}%)")

단계                         인원       전환율       도달률
----------------------------------------------------
Step 1 가입              677,075   100.00%   100.00%
Step 2 친구 형성           574,163    84.80%    84.80%
Step 3 투표 수신            10,647     1.85%     1.57%
Step 4 첫 결제              1,041     9.78%     0.15%

가입         → 친구 형성       102,912명 ( 15.2%)
친구 형성      → 투표 수신       563,516명 ( 98.1%)
투표 수신      → 첫 결제          9,606명 ( 90.2%)


In [9]:
직접 = step3 & set(pay['user_id'])
경유 = step3 & set(opened[opened > 0].index) & set(pay['user_id'])
우회 = 직접 - 경유

# 이들의 소비 기록
ph_우회 = ph[ph['user_id'].isin(우회) & (ph['delta_point'] < 0)]
print(f"우회 경로 결제자: {len(우회):,}명")
print(f"그중 소비 기록 있음: {ph_우회['user_id'].nunique():,}명")
print()
print(ph_우회['delta_point'].value_counts().sort_index())

우회 경로 결제자: 465명
그중 소비 기록 있음: 1명

delta_point
-10    12
Name: count, dtype: int64


In [10]:
ph['created_at'] = pd.to_datetime(ph['created_at'])
pay['created_at'] = pd.to_datetime(pay['created_at'])

# 첫 소비 시점
first_spend = ph[ph['delta_point'] < 0].groupby('user_id')['created_at'].min()
first_pay = pay.groupby('user_id')['created_at'].min()

common = first_spend.index.intersection(first_pay.index)
cmp = pd.DataFrame({
    '첫소비': first_spend.loc[common],
    '첫결제': first_pay.loc[common]
})
cmp['결제먼저'] = cmp['첫결제'] < cmp['첫소비']

print(f"둘 다 있는 사용자: {len(cmp):,}명")
print(f"결제가 먼저: {cmp['결제먼저'].sum():,}명 ({cmp['결제먼저'].mean()*100:.1f}%)")
print(f"소비가 먼저: {(~cmp['결제먼저']).sum():,}명")

둘 다 있는 사용자: 399명
결제가 먼저: 24명 (6.0%)
소비가 먼저: 375명


In [14]:
step1 = set(user['id'])

fr_a = fr[fr['status'] == 'A']
friends = pd.concat([fr_a['send_user_id'], fr_a['receive_user_id']]).value_counts()
step2 = set(friends.index) & step1

step3 = step2 & set(uqr['chosen_user_id'])

step4 = step3 & set(ph[ph['delta_point'] < 0]['user_id'])

step5 = step4 & set(pay['user_id'])

steps = [('가입', step1), ('친구 관계 형성', step2), ('투표 수신', step3),
         ('재화 소모', step4), ('첫 결제', step5)]

print("=== 1차 행동 루프 ===")
prev = None
for i, (name, s) in enumerate(steps):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    reach = f"{len(s)/len(step1)*100:7.2f}%"
    print(f"Step {i+1} {name:14s} {len(s):>9,} {conv:>9s} {reach:>9s}")
    prev = s

print()
for i in range(len(steps)-1):
    a, b = len(steps[i][1]), len(steps[i+1][1])
    print(f"{steps[i][0]:14s} → {steps[i+1][0]:14s} {a-b:>8,}명 ({(1-b/a)*100:5.1f}%)")

=== 1차 행동 루프 ===
Step 1 가입               677,075   100.00%   100.00%
Step 2 친구 관계 형성         574,163    84.80%    84.80%
Step 3 투표 수신             10,647     1.85%     1.57%
Step 4 재화 소모              3,367    31.62%     0.50%
Step 5 첫 결제                 358    10.63%     0.05%

가입             → 친구 관계 형성        102,912명 ( 15.2%)
친구 관계 형성       → 투표 수신           563,516명 ( 98.1%)
투표 수신          → 재화 소모             7,280명 ( 68.4%)
재화 소모          → 첫 결제              3,009명 ( 89.4%)


In [15]:
ph['created_at'] = pd.to_datetime(ph['created_at'])
pay['created_at'] = pd.to_datetime(pay['created_at'])
uqr['created_at'] = pd.to_datetime(uqr['created_at'])

# 첫 결제 시점
first_pay = pay.groupby('user_id')['created_at'].min()

# 2차 루프 시작점 = 1차 루프를 완주한 사용자
loop2_start = step5

print(f"2차 루프 시작 (첫 결제자): {len(loop2_start):,}명")

2차 루프 시작 (첫 결제자): 358명


In [21]:
# ============================================
# 데이터 로드
# ============================================
user = pd.read_csv(DATA / "accounts_user_preprocessed_2.csv")
pay  = pd.read_csv(DATA / "accounts_paymenthistory_preprocessed_2.csv")
ph   = pd.read_csv(DATA / "accounts_pointhistory_preprocessed_2.csv")
uqr  = pd.read_csv(DATA / "accounts_userquestionrecord_preprocessed_2.csv")
fr   = pd.read_csv(DATA / "accounts_friendrequest_preprocessed_2.csv",
                   usecols=['send_user_id','receive_user_id','status'])

pay['created_at'] = pd.to_datetime(pay['created_at'])
ph['created_at']  = pd.to_datetime(ph['created_at'])
uqr['created_at'] = pd.to_datetime(uqr['created_at'])

ph_spend = ph[ph['delta_point'] < 0]


# ============================================
# 1차 행동 루프
# ============================================
step1 = set(user['id'])

fr_a = fr[fr['status'] == 'A']
friends = pd.concat([fr_a['send_user_id'], fr_a['receive_user_id']]).value_counts()
step2 = set(friends.index) & step1

step3 = step2 & set(uqr['chosen_user_id'])
step4 = step3 & set(ph_spend['user_id'])
step5 = step4 & set(pay['user_id'])

print("=" * 58)
print("1차 행동 루프")
print("=" * 58)
steps1 = [('가입', step1), ('친구 관계 형성', step2), ('투표 수신', step3),
          ('재화 소모', step4), ('첫 결제', step5)]
prev = None
for i, (name, s) in enumerate(steps1):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    print(f"Step {i+1} {name:14s} {len(s):>9,} {conv:>9s}")
    prev = s


# ============================================
# 2차 행동 루프 — 시점 순서 엄격 적용
# ============================================
first_pay = pay.groupby('user_id')['created_at'].min()

# Step 6 — 첫 결제 이후 투표 수신 (사용자별 최초)
uqr_s = uqr[['chosen_user_id', 'created_at']].rename(
    columns={'chosen_user_id': 'user_id', 'created_at': 'vote_after_payment_at'})
uqr_s = uqr_s[uqr_s['user_id'].isin(step5)]
uqr_s['first_pay'] = uqr_s['user_id'].map(first_pay)
uqr_s = uqr_s[uqr_s['vote_after_payment_at'] > uqr_s['first_pay']]

stage6 = (uqr_s.sort_values('vote_after_payment_at')
               .drop_duplicates('user_id')[['user_id', 'vote_after_payment_at']])
step6 = set(stage6['user_id'])

# Step 7 — 재투표 이후 재화 소모 (사용자별 최초)
ph_s = ph_spend[['user_id', 'created_at']].rename(
    columns={'created_at': 'spend_after_vote_at'})

stage7 = stage6.merge(ph_s, on='user_id', how='inner')
stage7 = stage7[stage7['spend_after_vote_at'] > stage7['vote_after_payment_at']]
stage7 = (stage7.sort_values('spend_after_vote_at')
                .drop_duplicates('user_id')[['user_id', 'spend_after_vote_at']])
step7 = set(stage7['user_id'])

# Step 8 — 소비 이후 재결제 (사용자별 최초)
pay_s = pay[['user_id', 'created_at']].rename(columns={'created_at': 'repurchase_at'})

stage8 = stage7.merge(pay_s, on='user_id', how='inner')
stage8 = stage8[stage8['repurchase_at'] > stage8['spend_after_vote_at']]
stage8 = stage8.sort_values('repurchase_at').drop_duplicates('user_id')
step8 = set(stage8['user_id'])

print()
print("=" * 58)
print("2차 행동 루프")
print("=" * 58)
steps2 = [('첫 결제', step5), ('투표 수신', step6),
          ('재화 소모', step7), ('재결제', step8)]
prev = None
for i, (name, s) in enumerate(steps2):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    print(f"Step {i+5} {name:14s} {len(s):>9,} {conv:>9s}")
    prev = s

print()
for i in range(len(steps2)-1):
    a, b = len(steps2[i][1]), len(steps2[i+1][1])
    print(f"{steps2[i][0]:12s} → {steps2[i+1][0]:12s} {a-b:>7,}명 ({(1-b/a)*100:5.1f}%)")


# ============================================
# 1차 vs 2차 전환율 비교
# ============================================
print()
print("=" * 58)
print("1차 vs 2차 — 재화 소모 → 결제 전환율")
print("=" * 58)
print(f"1차 (첫 결제):  {len(step5)/len(step4)*100:6.2f}%  ({len(step5):,} / {len(step4):,})")
print(f"2차 (재결제):   {len(step8)/len(step7)*100:6.2f}%  ({len(step8):,} / {len(step7):,})")
print(f"배수:           {(len(step8)/len(step7))/(len(step5)/len(step4)):.2f}배")

1차 행동 루프
Step 1 가입               677,075   100.00%
Step 2 친구 관계 형성         574,163    84.80%
Step 3 투표 수신             10,647     1.85%
Step 4 재화 소모              3,367    31.62%
Step 5 첫 결제                 358    10.63%

2차 행동 루프
Step 5 첫 결제                 358   100.00%
Step 6 투표 수신                353    98.60%
Step 7 재화 소모                334    94.62%
Step 8 재결제                   71    21.26%

첫 결제         → 투표 수신              5명 (  1.4%)
투표 수신        → 재화 소모             19명 (  5.4%)
재화 소모        → 재결제              263명 ( 78.7%)

1차 vs 2차 — 재화 소모 → 결제 전환율
1차 (첫 결제):   10.63%  (358 / 3,367)
2차 (재결제):    21.26%  (71 / 334)
배수:           2.00배


In [22]:
import pandas as pd
from pathlib import Path

DATA = PROJECT_ROOT / "data" / "processed" / "preprocessed_data_2"

# ============================================
# 1. 데이터 로드
# ============================================
user = pd.read_csv(DATA / "accounts_user_preprocessed_2.csv")
pay  = pd.read_csv(DATA / "accounts_paymenthistory_preprocessed_2.csv")
ph   = pd.read_csv(DATA / "accounts_pointhistory_preprocessed_2.csv")
uqr  = pd.read_csv(DATA / "accounts_userquestionrecord_preprocessed_2.csv")
fr   = pd.read_csv(DATA / "accounts_friendrequest_preprocessed_2.csv",
                   usecols=['send_user_id', 'receive_user_id', 'status'])

pay['created_at'] = pd.to_datetime(pay['created_at'])
ph['created_at']  = pd.to_datetime(ph['created_at'])
uqr['created_at'] = pd.to_datetime(uqr['created_at'])

ph_spend = ph[ph['delta_point'] < 0]

print(f"user {len(user):,} / pay {len(pay):,} / ph {len(ph):,} / "
      f"uqr {len(uqr):,} / fr {len(fr):,}")


# ============================================
# 2. 1차 행동 루프
# ============================================
step1 = set(user['id'])

fr_a = fr[fr['status'] == 'A']
friends = pd.concat([fr_a['send_user_id'], fr_a['receive_user_id']]).value_counts()
step2 = set(friends.index) & step1

step3 = step2 & set(uqr['chosen_user_id'])
step4 = step3 & set(ph_spend['user_id'])
step5 = step4 & set(pay['user_id'])

print()
print("=" * 58)
print("1차 행동 루프")
print("=" * 58)
steps1 = [('가입', step1), ('친구 관계 형성', step2), ('투표 수신', step3),
          ('재화 소모', step4), ('첫 결제', step5)]
prev = None
for i, (name, s) in enumerate(steps1):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    reach = f"{len(s)/len(step1)*100:7.2f}%"
    print(f"Step {i+1} {name:14s} {len(s):>9,} {conv:>9s} {reach:>9s}")
    prev = s

print()
for i in range(len(steps1) - 1):
    a, b = len(steps1[i][1]), len(steps1[i+1][1])
    print(f"{steps1[i][0]:14s} → {steps1[i+1][0]:14s} {a-b:>8,}명 ({(1-b/a)*100:5.1f}%)")


# ============================================
# 3. 2차 행동 루프 — 시점 순서 엄격 적용
#    첫 결제 < 투표 수신 < 재화 소모 < 재결제
# ============================================
first_pay = pay.groupby('user_id')['created_at'].min()

# Step 6 — 첫 결제 이후 최초 투표 수신
v = uqr.loc[uqr['chosen_user_id'].isin(step5), ['chosen_user_id', 'created_at']]
v = v.rename(columns={'chosen_user_id': 'user_id', 'created_at': 'vote_at'})
v['first_pay'] = v['user_id'].map(first_pay)
v = v[v['vote_at'] > v['first_pay']]

stage6 = v.sort_values('vote_at').drop_duplicates('user_id')[['user_id', 'vote_at']]
step6 = set(stage6['user_id'])

# Step 7 — 투표 수신 이후 최초 재화 소모
s = ph_spend[['user_id', 'created_at']].rename(columns={'created_at': 'spend_at'})
stage7 = stage6.merge(s, on='user_id', how='inner')
stage7 = stage7[stage7['spend_at'] > stage7['vote_at']]
stage7 = stage7.sort_values('spend_at').drop_duplicates('user_id')[
    ['user_id', 'vote_at', 'spend_at']]
step7 = set(stage7['user_id'])

# Step 8 — 재화 소모 이후 최초 결제 (= 재결제)
p = pay[['user_id', 'created_at']].rename(columns={'created_at': 'pay_at'})
stage8 = stage7.merge(p, on='user_id', how='inner')
stage8 = stage8[stage8['pay_at'] > stage8['spend_at']]
stage8 = stage8.sort_values('pay_at').drop_duplicates('user_id')
step8 = set(stage8['user_id'])

print()
print("=" * 58)
print("2차 행동 루프")
print("=" * 58)
steps2 = [('첫 결제', step5), ('투표 수신', step6),
          ('재화 소모', step7), ('재결제', step8)]
prev = None
for i, (name, s) in enumerate(steps2):
    conv = f"{len(s)/len(prev)*100:7.2f}%" if prev else "  100.00%"
    print(f"Step {i+5} {name:14s} {len(s):>9,} {conv:>9s}")
    prev = s

print()
for i in range(len(steps2) - 1):
    a, b = len(steps2[i][1]), len(steps2[i+1][1])
    print(f"{steps2[i][0]:12s} → {steps2[i+1][0]:12s} {a-b:>7,}명 ({(1-b/a)*100:5.1f}%)")


# ============================================
# 4. 검증
# ============================================
print()
print("=" * 58)
print("검증")
print("=" * 58)
print(f"step2 ⊆ step1: {step2 <= step1}")
print(f"step3 ⊆ step2: {step3 <= step2}")
print(f"step4 ⊆ step3: {step4 <= step3}")
print(f"step5 ⊆ step4: {step5 <= step4}")
print(f"step6 ⊆ step5: {step6 <= step5}")
print(f"step7 ⊆ step6: {step7 <= step6}")
print(f"step8 ⊆ step7: {step8 <= step7}")

chk = stage8.merge(first_pay.rename('fp'), left_on='user_id', right_index=True)
print()
print("시점 순서 위반 (전부 0이어야 정상)")
print(f"  first_pay >= vote_at: {(chk['fp']       >= chk['vote_at']).sum()}")
print(f"  vote_at   >= spend_at: {(chk['vote_at']  >= chk['spend_at']).sum()}")
print(f"  spend_at  >= pay_at:   {(chk['spend_at'] >= chk['pay_at']).sum()}")


# ============================================
# 5. 1차 vs 2차 전환율 비교
# ============================================
print()
print("=" * 58)
print("1차 vs 2차 — 재화 소모 → 결제 전환율")
print("=" * 58)
r1 = len(step5) / len(step4)
r2 = len(step8) / len(step7)
print(f"1차 (첫 결제):  {r1*100:6.2f}%  ({len(step5):,} / {len(step4):,})")
print(f"2차 (재결제):   {r2*100:6.2f}%  ({len(step8):,} / {len(step7):,})")
print(f"배수:           {r2/r1:.2f}배")


# ============================================
# 6. 커버리지 참고
# ============================================
print()
print("=" * 58)
print("테이블 커버리지")
print("=" * 58)
total = len(user)
for name, n in [
    ('accounts_user',              total),
    ('friendrequest (양방향)',      len(set(fr_a['send_user_id']) | set(fr_a['receive_user_id']))),
    ('paymenthistory',             pay['user_id'].nunique()),
    ('uqr (수신자)',                uqr['chosen_user_id'].nunique()),
    ('pointhistory',               ph['user_id'].nunique()),
]:
    print(f"{name:26s} {n:>9,}명  {n/total*100:6.2f}%")

user 677,075 / pay 95,137 / ph 88,762 / uqr 800,793 / fr 7,793,140

1차 행동 루프
Step 1 가입               677,075   100.00%   100.00%
Step 2 친구 관계 형성         574,163    84.80%    84.80%
Step 3 투표 수신             10,647     1.85%     1.57%
Step 4 재화 소모              3,367    31.62%     0.50%
Step 5 첫 결제                 358    10.63%     0.05%

가입             → 친구 관계 형성        102,912명 ( 15.2%)
친구 관계 형성       → 투표 수신           563,516명 ( 98.1%)
투표 수신          → 재화 소모             7,280명 ( 68.4%)
재화 소모          → 첫 결제              3,009명 ( 89.4%)

2차 행동 루프
Step 5 첫 결제                 358   100.00%
Step 6 투표 수신                353    98.60%
Step 7 재화 소모                334    94.62%
Step 8 재결제                   71    21.26%

첫 결제         → 투표 수신              5명 (  1.4%)
투표 수신        → 재화 소모             19명 (  5.4%)
재화 소모        → 재결제              263명 ( 78.7%)

검증
step2 ⊆ step1: True
step3 ⊆ step2: True
step4 ⊆ step3: True
step5 ⊆ step4: True
step6 ⊆ step5: True
step7 ⊆ step6: True
step8 ⊆ step7: Tr